# Topic Modeling on `body_anonimizado` — merged_df1

Pipeline:
1. Reconstruct **merged_df1** (conversations + tipificaciones LEFT JOIN)
2. Merge with the anonymized text (`body_anonimizado`)
3. Group messages into conversations by `OpenchannelInteractionId`
4. **LDA** topic modeling on anonymized messages
5. **BERT + LDA ensemble** risk scoring per message
6. Conversation-level topic & risk aggregation
7. Visualizations

---
**Files expected** (same names used across the project):
- `30.01.25 Uic.xlsx` — main export (sheet: `Export`)
- `20.02.25TipificacionesConversaciones.xlsx` — tipifications
- `MOSTRA_1_anonimizado.xlsx` — anonymized output with `body_anonimizado` column

## 0. Imports

In [ ]:
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, fbeta_score, precision_recall_curve,
    confusion_matrix, roc_auc_score
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.utils.class_weight import compute_class_weight

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load Source Files and Build merged_df1

Three-way build:
1. `df_principal` (xats) × `df_tipificaciones` → **merged_df1**  
2. merged_df1 × `df_fichas_crm` → **merged_df_full**

Join keys match the BigQuery SQL in `query_FAE.sql`:
- xats ↔ tipificaciones: `OpenchannelInteractionId = id` AND `ContactId`
- merged1 ↔ fichas: `ContactId` + `fecha` + `conversacion_dia` (DENSE_RANK)

In [ ]:
# ------------------------------------------------------------------
# Colab paths: '/content/30.01.25 Uic.xlsx', etc.
# ------------------------------------------------------------------
PATH_PRINCIPAL      = '30.01.25 Uic.xlsx'
PATH_TIPIFICACIONES = '20.02.25TipificacionesConversaciones.xlsx'
PATH_FICHAS_CRM     = '20.02.25FichasCrm (2).xlsx'
PATH_ANONIMIZADO    = 'MOSTRA_1_anonimizado.xlsx'

df_principal      = pd.read_excel(PATH_PRINCIPAL, sheet_name='Export')
df_tipificaciones = pd.read_excel(PATH_TIPIFICACIONES)
df_fichas_crm     = pd.read_excel(PATH_FICHAS_CRM)

# Normalise ContactId column name in fichas (Catalan column name)
crm_phone_col = next(
    (c for c in df_fichas_crm.columns if 'telèfon' in c.lower() or 'telefon' in c.lower()),
    None
)
if crm_phone_col:
    df_fichas_crm = df_fichas_crm.rename(columns={crm_phone_col: 'ContactId_crm'})

print(f'df_principal      : {df_principal.shape}')
print(f'  columns: {df_principal.columns.tolist()}')
print(f'\ndf_tipificaciones : {df_tipificaciones.shape}')
print(f'  columns: {df_tipificaciones.columns.tolist()}')
print(f'\ndf_fichas_crm     : {df_fichas_crm.shape}')
print(f'  columns: {df_fichas_crm.columns.tolist()}')

In [ ]:
# ── Step 1: xats × tipificaciones → merged_df1 ──────────────────
df_tipificaciones = df_tipificaciones.rename(columns={'ContactId': 'ContactId1'})

merged_df1 = pd.merge(
    df_principal,
    df_tipificaciones,
    how='left',
    left_on=['OpenchannelInteractionId', 'ContactId'],
    right_on=['id', 'ContactId1'],
    suffixes=('', '_tip')
)
# Drop the redundant id from tipificaciones
merged_df1 = merged_df1.drop(columns=[c for c in merged_df1.columns if c == 'id_tip'], errors='ignore')

print(f'merged_df1 shape : {merged_df1.shape}')

# ── Step 2: add temporal features needed for fichas join (from query_FAE.sql) ──
merged_df1['createdAt'] = pd.to_datetime(merged_df1['createdAt'], errors='coerce')
merged_df1['fecha']     = merged_df1['createdAt'].dt.normalize()  # date only

# DENSE_RANK of conversation per (ContactId, date) — mirrors conversacion_dia1 in SQL
merged_df1['conversacion_dia'] = (
    merged_df1.sort_values('createdAt')
    .groupby(['ContactId', 'fecha'])['createdAt']
    .rank(method='dense')
    .astype(int)
)

# ── Step 3: prepare fichas for join ────────────────────────────────
df_fichas_crm['Data/hora inici'] = pd.to_datetime(
    df_fichas_crm['Data/hora inici'], errors='coerce', dayfirst=True
)
df_fichas_crm['fecha']            = df_fichas_crm['Data/hora inici'].dt.normalize()
df_fichas_crm['conversacion_dia'] = (
    df_fichas_crm.sort_values('Data/hora inici')
    .groupby(['ContactId', 'fecha'])['Data/hora inici']
    .rank(method='dense')
    .astype(int)
)

# ── Step 4: merged_df1 × fichas → merged_df_full ───────────────────
merged_df_full = pd.merge(
    merged_df1,
    df_fichas_crm.add_suffix('_crm').rename(columns={
        'ContactId_crm': 'ContactId',
        'fecha_crm':     'fecha',
        'conversacion_dia_crm': 'conversacion_dia'
    }),
    how='left',
    on=['ContactId', 'fecha', 'conversacion_dia'],
    suffixes=('', '_dup')
)
merged_df_full = merged_df_full.loc[:, ~merged_df_full.columns.str.endswith('_dup')]

print(f'merged_df_full shape : {merged_df_full.shape}')
print(f'Fichas match rate     : {merged_df_full["Data/hora inici_crm"].notna().mean():.1%}')
merged_df_full.head(2)

## 2. Attach body_anonimizado

In [ ]:
df_anon = pd.read_excel(PATH_ANONIMIZADO)
print(f'df_anon: {df_anon.shape}  columns: {df_anon.columns.tolist()}')

if 'body_anonimizado' not in merged_df_full.columns:
    merged_df_full = pd.merge(
        merged_df_full,
        df_anon[['id', 'body_anonimizado']],
        on='id', how='left'
    )

merged_df_full = merged_df_full.dropna(subset=['body_anonimizado']).reset_index(drop=True)
print(f'Rows with body_anonimizado: {len(merged_df_full)}')

In [ ]:
# Keep only needed cols for downstream (avoids column explosion)
BASE_COLS = [
    'id', 'direction', 'createdAt', 'fecha', 'conversacion_dia',
    'OpenchannelInteractionId', 'ContactId', 'UserId',
    'body_anonimizado',
    'secret',          # flagged sensitive message
    'Score',           # conversation quality score
    'Status',          # conversation status
    'sentBy',          # who sent (agent / bot / user)
]
# Add tipification columns that made it through (all non-null cols from df_tipificaciones)
tip_extra = [c for c in merged_df_full.columns
             if c not in BASE_COLS and c in df_tipificaciones.columns
             and c not in ('id', 'ContactId1')]
# Add fichas CRM columns (suffixed _crm)
crm_extra = [c for c in merged_df_full.columns if c.endswith('_crm')]

keep = [c for c in BASE_COLS + tip_extra + crm_extra if c in merged_df_full.columns]
df = merged_df_full[keep].copy()

print(f'Working dataframe: {df.shape}')
print(f'Columns ({len(df.columns)}): {df.columns.tolist()}')

## 3. Preprocessing

In [ ]:
def preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'[^a-záéíóúüñ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text_clean'] = df['body_anonimizado'].apply(preprocess)

# User messages only for NLP branches
df_in  = df[df['direction'] == 'in'].copy().reset_index(drop=True)
df_out = df[df['direction'] == 'out'].copy()

print(f"Messages 'in'  (user) : {len(df_in)}")
print(f"Messages 'out' (agent): {len(df_out)}")

## 3b. Structured Feature Engineering

Variables extracted from every data source to complement BERT+LDA and reduce False Negatives.

| Group | Features | Source |
|-------|----------|--------|
| Temporal | hour, is_night, is_weekend, conversacion_dia | xats `createdAt` |
| Message | char_len, word_count, uppercase_ratio, question_marks, exclamation_marks | xats `body_anonimizado` |
| Conversation | n_msgs_in_conv, secret_count, secret_ratio, score_mean, score_min | xats `secret`/`Score` |
| Tipification | one-hot of tipification category | tipificaciones |
| Fichas CRM | n_fichas_per_contact, hora_fichas_sin/cos | fichas CRM |

In [ ]:
# ── 3b-1. Temporal features (from xats createdAt + SQL-derived fields) ──────

df_in['hour']       = df_in['createdAt'].dt.hour
df_in['hour_sin']   = np.sin(2 * np.pi * df_in['hour'] / 24)
df_in['hour_cos']   = np.cos(2 * np.pi * df_in['hour'] / 24)
df_in['is_night']   = df_in['hour'].between(23, 23) | df_in['hour'].between(0, 6)
df_in['is_night']   = df_in['is_night'].astype(int)
df_in['is_weekend'] = df_in['createdAt'].dt.dayofweek.isin([5, 6]).astype(int)
# conversacion_dia already present: how many contacts that day (higher → escalation)

# ── 3b-2. Message-level text features ───────────────────────────────────────

raw_body = df_in['body_anonimizado'].astype(str)

df_in['char_len']        = raw_body.str.len()
df_in['word_count']      = raw_body.str.split().str.len()
df_in['uppercase_ratio'] = raw_body.apply(
    lambda t: sum(1 for c in t if c.isupper()) / max(len(t), 1)
)
df_in['n_question']      = raw_body.str.count(r'\?')
df_in['n_exclamation']   = raw_body.str.count(r'!')
df_in['n_ellipsis']      = raw_body.str.count(r'\.{3}|…')   # trailing off mid-sentence

# ── 3b-3. secret flag (binary from xats) ────────────────────────────────────

df_in['secret_flag'] = pd.to_numeric(df_in.get('secret', 0), errors='coerce').fillna(0).astype(int)

# ── 3b-4. Score (normalised, fill missing with median) ──────────────────────

if 'Score' in df_in.columns:
    median_score         = df_in['Score'].median()
    df_in['score_norm']  = df_in['Score'].fillna(median_score)
    # Invert: lower score may signal distress
    df_in['score_inv']   = 1 - (df_in['score_norm'] / df_in['score_norm'].max().clip(lower=1e-9))
else:
    df_in['score_norm'] = 0.5
    df_in['score_inv']  = 0.5

print('Temporal + message features added.')
df_in[['hour', 'is_night', 'is_weekend', 'conversacion_dia',
       'char_len', 'word_count', 'secret_flag', 'score_inv']].describe()

In [ ]:
# ── 3b-5. Conversation-level aggregates (joined back per message) ────────────

conv_agg = (
    df_in.groupby('OpenchannelInteractionId')
    .agg(
        conv_n_msgs       = ('id',           'count'),
        conv_secret_count = ('secret_flag',  'sum'),
        conv_score_mean   = ('score_norm',   'mean'),
        conv_score_min    = ('score_norm',   'min'),
        conv_night_msgs   = ('is_night',     'sum'),
        conv_max_day_rank = ('conversacion_dia', 'max'),   # how many contacts that day
    )
    .reset_index()
)
conv_agg['conv_secret_ratio'] = (
    conv_agg['conv_secret_count'] / conv_agg['conv_n_msgs'].clip(lower=1)
)

df_in = df_in.merge(conv_agg, on='OpenchannelInteractionId', how='left')
print('Conversation-level features added.')

# ── 3b-6. Tipification one-hot ───────────────────────────────────────────────

tip_cols_present = [c for c in tip_extra if c in df_in.columns]
if tip_cols_present:
    print(f'Tipification columns: {tip_cols_present}')
    for col in tip_cols_present:
        df_in[col] = df_in[col].astype(str).fillna('unknown')
    tip_dummies = pd.get_dummies(df_in[tip_cols_present], prefix=tip_cols_present, drop_first=True)
    df_in = pd.concat([df_in, tip_dummies], axis=1)
    tip_feature_cols = tip_dummies.columns.tolist()
    print(f'  → {len(tip_feature_cols)} one-hot columns')
else:
    tip_feature_cols = []
    print('No tipification columns found in df_in.')

# ── 3b-7. Fichas CRM features ────────────────────────────────────────────────

# Number of CRM fichas per ContactId (proxy for repeat contacts / chronic users)
fichas_per_contact = (
    df_fichas_crm.groupby('ContactId').size()
    .reset_index(name='n_fichas_contact')
)
df_in = df_in.merge(fichas_per_contact, on='ContactId', how='left')
df_in['n_fichas_contact'] = df_in['n_fichas_contact'].fillna(0).astype(int)

# Hour from fichas start time (cyclic encoding)
crm_dt_col = 'Data/hora inici_crm'
if crm_dt_col in df_in.columns:
    crm_hour = pd.to_datetime(df_in[crm_dt_col], errors='coerce').dt.hour.fillna(12)
    df_in['fichas_hour_sin'] = np.sin(2 * np.pi * crm_hour / 24)
    df_in['fichas_hour_cos'] = np.cos(2 * np.pi * crm_hour / 24)
    crm_feat_cols = ['fichas_hour_sin', 'fichas_hour_cos']
else:
    crm_feat_cols = []

print(f'Fichas features: n_fichas_contact + {crm_feat_cols}')

# ── Summary of all structured feature columns ────────────────────────────────

STRUCT_FEAT_COLS = [
    'hour_sin', 'hour_cos', 'is_night', 'is_weekend', 'conversacion_dia',
    'char_len', 'word_count', 'uppercase_ratio', 'n_question', 'n_exclamation', 'n_ellipsis',
    'secret_flag', 'score_inv',
    'conv_n_msgs', 'conv_secret_ratio', 'conv_score_mean', 'conv_score_min',
    'conv_night_msgs', 'conv_max_day_rank',
    'n_fichas_contact',
] + crm_feat_cols + tip_feature_cols

STRUCT_FEAT_COLS = [c for c in STRUCT_FEAT_COLS if c in df_in.columns]
print(f'\nTotal structured features: {len(STRUCT_FEAT_COLS)}')

## 4. LDA Topic Modeling on body_anonimizado

We run LDA on **user messages** (`direction='in'`) — these carry the relevant clinical content.

In [ ]:
N_TOPICS = 6   # Adjust based on your domain knowledge

SPANISH_STOP_WORDS = [
    'de','la','el','en','y','a','los','del','se','las','por','un',
    'para','con','una','su','al','lo','como','más','pero','sus','le',
    'ya','o','este','si','porque','esta','entre','cuando','muy','sin',
    'sobre','también','me','hasta','hay','donde','quien','desde','todo',
    'nos','durante','todos','uno','les','ni','contra','otros','ese',
    'eso','ante','ellos','e','esto','mí','antes','algunos','qué','unos',
    'yo','otro','otras','otras','tanto','esa','estos','mucho','quienes',
    'nada','muchos','cual','poco','ella','estar','estas','mi','ha','vez',
    'hola','buenas','gracias','ok','vale','sí','no','que','te','es',
    'nombre'
]

vec = CountVectorizer(
    stop_words=SPANISH_STOP_WORDS,
    max_df=0.90,
    min_df=2,
    max_features=1000
)

texts_in = df_in['text_clean'].tolist()
dtm = vec.fit_transform(texts_in)
vocab = vec.get_feature_names_out()

print(f'Vocabulary size : {len(vocab)}')
print(f'DTM shape       : {dtm.shape}')

In [ ]:
lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    max_iter=30,
    learning_method='batch',
    random_state=42
)
lda.fit(dtm)

# Topic-probability matrix for all user messages
doc_topic_matrix = lda.transform(dtm)   # shape (n_messages, N_TOPICS)

print('\nTop 12 words per topic:')
for idx, topic in enumerate(lda.components_):
    top_words = [vocab[i] for i in topic.argsort()[-12:][::-1]]
    print(f'  Topic {idx}: {", ".join(top_words)}')

In [ ]:
# Attach topic probabilities and dominant topic to df_in
for t in range(N_TOPICS):
    df_in[f'topic_{t}'] = doc_topic_matrix[:, t]

df_in['dominant_topic'] = doc_topic_matrix.argmax(axis=1)

df_in[['id', 'body_anonimizado', 'dominant_topic'] +
      [f'topic_{t}' for t in range(N_TOPICS)]].head(8)

## 5. Conversation-Level Topic Aggregation

Group messages by `OpenchannelInteractionId` and compute the **mean topic distribution** per conversation.

In [ ]:
topic_cols = [f'topic_{t}' for t in range(N_TOPICS)]

conv_topics = (
    df_in
    .groupby('OpenchannelInteractionId')[topic_cols]
    .mean()
    .reset_index()
)
conv_topics['dominant_topic'] = conv_topics[topic_cols].values.argmax(axis=1)
conv_topics['n_messages'] = (
    df_in.groupby('OpenchannelInteractionId').size().values
)

print(f'Conversations: {len(conv_topics)}')
conv_topics.sort_values('n_messages', ascending=False).head(8)

## 6. BERT + LDA Ensemble — Risk Scoring

We use the same two-branch architecture as the suicide-risk ensemble notebook,
now applied directly to `body_anonimizado` messages.

In [ ]:
# ── Feature matrix: BERT (768) + LDA (N_TOPICS) + Structured ────────────────
X_lda    = doc_topic_matrix   # (n_msgs, N_TOPICS)

X_struct = df_in[STRUCT_FEAT_COLS].fillna(0).values.astype(float)

X_combined = np.hstack([X_bert, X_lda, X_struct])

# Scale
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_combined)

print(f'Feature matrix breakdown:')
print(f'  BERT        : {X_bert.shape[1]:>5d}')
print(f'  LDA topics  : {X_lda.shape[1]:>5d}')
print(f'  Structured  : {X_struct.shape[1]:>5d}')
print(f'  ─────────────────────')
print(f'  Total       : {X_combined.shape[1]:>5d}')

### 6a. Unsupervised risk proxy (cosine similarity to risk-seed centroid)

Used when no labelled data is available.

In [ ]:
print('Encoding messages with BERT (this may take a few minutes) ...')
X_bert = bert_embeddings(df_in['text_clean'].tolist())
print(f'BERT embeddings shape: {X_bert.shape}')  # (n_messages, 768)

### 6b. Supervised Ensemble — F2-optimised (requires `risk_label` column)

**F2 score** (β=2) weights recall twice as much as precision — the right choice when
a missed risk case (False Negative) is more costly than a false alarm.

Strategy:
1. Train `LogisticRegression(class_weight='balanced')` on the full feature matrix
2. Evaluate with `fbeta_score(beta=2)` and cross-validation
3. Sweep decision threshold on [0.1, 0.9] and pick the threshold that maximises F2
4. Report classification report at the optimal threshold

In [ ]:
# ── 6b-1. Setup ─────────────────────────────────────────────────────────────

# Set SUPERVISED = True once you have a risk_label column in merged_df_full
# The column must be binary: 1 = suicide risk, 0 = no risk
SUPERVISED    = 'risk_label' in df_in.columns
LABEL_COL     = 'risk_label'

if not SUPERVISED:
    print('SUPERVISED=False — no risk_label column found.')
    print('Using bert_risk_score as unsupervised proxy instead.')
    print('To activate: add a binary risk_label column to your data.')
else:
    y = df_in[LABEL_COL].values
    print(f'Labels: {pd.Series(y).value_counts().to_dict()}')

In [ ]:
if SUPERVISED:
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import make_scorer

    # ── 6b-2. Train/test split ───────────────────────────────────────────────
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42, stratify=y
    )

    # ── 6b-3. Balanced logistic regression ──────────────────────────────────
    meta_clf = LogisticRegression(
        C=1.0,
        max_iter=2000,
        class_weight='balanced',   # upweights the minority (risk) class
        random_state=42,
        solver='lbfgs'
    )
    meta_clf.fit(X_tr, y_tr)

    # ── 6b-4. Cross-validated F2 on training set ────────────────────────────
    f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)
    cv_f2 = cross_val_score(
        LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced',
                           random_state=42, solver='lbfgs'),
        X_tr, y_tr,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring=f2_scorer
    )
    print(f'5-fold CV F2 (train): {cv_f2.mean():.3f} ± {cv_f2.std():.3f}')

    # ── 6b-5. Threshold sweep to maximise F2 ────────────────────────────────
    y_prob_te = meta_clf.predict_proba(X_te)[:, 1]

    thresholds  = np.linspace(0.1, 0.9, 81)
    f2_scores   = [fbeta_score(y_te, (y_prob_te >= t).astype(int),
                               beta=2, zero_division=0) for t in thresholds]
    fn_counts   = [((y_te == 1) & (y_prob_te < t)).sum() for t in thresholds]

    best_idx   = int(np.argmax(f2_scores))
    best_thr   = thresholds[best_idx]
    best_f2    = f2_scores[best_idx]

    print(f'\nOptimal threshold : {best_thr:.2f}')
    print(f'F2 at threshold   : {best_f2:.3f}')
    print(f'FN at threshold   : {fn_counts[best_idx]}')

    y_pred_opt = (y_prob_te >= best_thr).astype(int)
    print('\nClassification Report at optimal threshold:')
    print(classification_report(y_te, y_pred_opt,
          target_names=['no_risk', 'suicide_risk'], zero_division=0))

    df_in['ensemble_risk_prob'] = meta_clf.predict_proba(X_scaled)[:, 1]
    df_in['ensemble_pred']      = (df_in['ensemble_risk_prob'] >= best_thr).astype(int)

In [ ]:
if SUPERVISED:
    # ── 6b-6. Threshold optimisation plot ───────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 7), sharex=True)

    ax1.plot(thresholds, f2_scores, color='steelblue', lw=2, label='F2 score')
    ax1.axvline(best_thr, color='tomato', linestyle='--',
                label=f'Optimal thr = {best_thr:.2f}  (F2={best_f2:.3f})')
    ax1.set_ylabel('F2 score')
    ax1.set_title('F2 Score vs Decision Threshold')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(thresholds, fn_counts, color='tomato', lw=2, label='False Negatives')
    ax2.axvline(best_thr, color='steelblue', linestyle='--',
                label=f'Optimal thr = {best_thr:.2f}')
    ax2.set_xlabel('Decision threshold')
    ax2.set_ylabel('False Negatives (missed risks)')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('threshold_f2_fn.png', dpi=120, bbox_inches='tight')
    plt.show()

    # ── 6b-7. Confusion matrix at optimal threshold ─────────────────────────
    cm = confusion_matrix(y_te, y_pred_opt)
    fig, ax = plt.subplots(figsize=(4, 3.5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['no_risk', 'suicide_risk'])
    ax.set_yticks([0, 1]); ax.set_yticklabels(['no_risk', 'suicide_risk'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix  (thr={best_thr:.2f})')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)
    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig('confusion_matrix_f2.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# Risk-seed phrases (Spanish)
RISK_SEEDS = [
    "no quiero seguir viviendo",
    "quiero hacerme daño",
    "no puedo más con mi vida",
    "pienso en suicidarme",
    "ya no tiene sentido vivir",
    "quiero que todo termine",
    "soy una carga para todos",
    "he pensado en quitarme la vida",
]

print('Encoding risk seed phrases ...')
seed_clean   = [preprocess(s) for s in RISK_SEEDS]
seed_emb     = bert_embeddings(seed_clean)             # (n_seeds, 768)
risk_centroid = seed_emb.mean(axis=0, keepdims=True)   # (1, 768)

# Cosine similarity between each message and the risk centroid
def cosine_sim(A, B):
    A_norm = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)
    B_norm = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-9)
    return (A_norm * B_norm).sum(axis=1)

risk_score = cosine_sim(X_bert, risk_centroid)  # (n_messages,)

# Normalise to [0, 1]
risk_score_norm = (risk_score - risk_score.min()) / (risk_score.max() - risk_score.min() + 1e-9)

df_in = df_in.copy()
df_in['bert_risk_score'] = risk_score_norm

print(f'Risk score — mean: {risk_score_norm.mean():.3f}  max: {risk_score_norm.max():.3f}')

### 6b. (Optional) Supervised meta-classifier — requires labelled data

In [ ]:
# -----------------------------------------------------------------------
# If you have a 'risk_label' column in merged_df1 (0/1), uncomment this:
# -----------------------------------------------------------------------

# y = df_in['risk_label'].values
# meta_clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
# meta_clf.fit(X_scaled, y)
# df_in['ensemble_risk_prob'] = meta_clf.predict_proba(X_scaled)[:, 1]
# print(classification_report(y, meta_clf.predict(X_scaled),
#       target_names=['no_risk', 'suicide_risk']))

## 7. Conversation-Level Risk Aggregation

In [ ]:
conv_risk = (
    df_in
    .groupby('OpenchannelInteractionId')
    .agg(
        n_messages    = ('id', 'count'),
        mean_risk     = ('bert_risk_score', 'mean'),
        max_risk      = ('bert_risk_score', 'max'),
        dominant_topic= ('dominant_topic',  lambda x: x.mode()[0])
    )
    .reset_index()
    .sort_values('max_risk', ascending=False)
)

# Flag conversations above 75th-percentile max risk
threshold = conv_risk['max_risk'].quantile(0.75)
conv_risk['high_risk_flag'] = conv_risk['max_risk'] >= threshold

print(f'High-risk conversations (top 25%): {conv_risk["high_risk_flag"].sum()}')
conv_risk.head(10)

## 8. Visualizations

In [ ]:
# --- 8a. Top words per topic ---
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
colors = list(mcolors.TABLEAU_COLORS.values())

for idx, (ax, topic) in enumerate(zip(axes, lda.components_)):
    n = 12
    top_idx   = topic.argsort()[-n:][::-1]
    top_words = [vocab[i] for i in top_idx]
    top_vals  = topic[top_idx]
    top_vals  = top_vals / top_vals.sum()
    ax.barh(top_words[::-1], top_vals[::-1], color=colors[idx % len(colors)])
    ax.set_title(f'Topic {idx}', fontsize=11)
    ax.set_xlabel('Relative weight')

plt.suptitle('LDA — Top 12 Words per Topic (body_anonimizado)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('lda_top_words.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- 8b. Dominant topic distribution ---
topic_counts = df_in['dominant_topic'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(topic_counts.index, topic_counts.values,
       color=[colors[i % len(colors)] for i in topic_counts.index])
ax.set_xlabel('Topic')
ax.set_ylabel('Number of messages')
ax.set_title('Dominant Topic Distribution — user messages')
ax.set_xticks(range(N_TOPICS))
plt.tight_layout()
plt.savefig('dominant_topic_dist.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- 8c. Risk score distribution ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df_in['bert_risk_score'], bins=30, color='tomato', edgecolor='white', alpha=0.85)
ax.axvline(threshold, color='black', linestyle='--', label=f'P75 threshold = {threshold:.2f}')
ax.set_xlabel('BERT risk score (normalised)')
ax.set_ylabel('Messages')
ax.set_title('Distribution of Risk Scores (body_anonimizado)')
ax.legend()
plt.tight_layout()
plt.savefig('risk_score_dist.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- 8d. Heatmap — topic distribution per conversation (top 30 by message count) ---
top_convs = conv_topics.sort_values('n_messages', ascending=False).head(30)
heat_data = top_convs[topic_cols].values

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(heat_data, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(N_TOPICS))
ax.set_xticklabels([f'T{i}' for i in range(N_TOPICS)])
ax.set_yticks(range(len(top_convs)))
ax.set_yticklabels(top_convs['OpenchannelInteractionId'].astype(str), fontsize=7)
ax.set_xlabel('Topic')
ax.set_ylabel('Conversation ID')
ax.set_title('Topic Distribution — top 30 conversations')
plt.colorbar(im, ax=ax, label='Mean topic probability')
plt.tight_layout()
plt.savefig('conv_topic_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# --- 8e. Mean & max risk per conversation (top 40) ---
plot_df = conv_risk.head(40).sort_values('mean_risk', ascending=True)
y_pos   = range(len(plot_df))

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(y_pos, plot_df['max_risk'],  color='tomato',    alpha=0.6, label='Max risk')
ax.barh(y_pos, plot_df['mean_risk'], color='steelblue', alpha=0.8, label='Mean risk')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(plot_df['OpenchannelInteractionId'].astype(str), fontsize=7)
ax.set_xlabel('Risk score')
ax.set_title('Mean & Max Risk per Conversation')
ax.legend()
plt.tight_layout()
plt.savefig('conv_risk_scores.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Export Results

In [ ]:
# Message-level output
msg_output_cols = [
    'id', 'OpenchannelInteractionId', 'ContactId',
    'direction', 'body_anonimizado',
    'dominant_topic', 'bert_risk_score'
] + topic_cols

df_in[msg_output_cols].to_excel('messages_topics_risk.xlsx', index=False)

# Conversation-level output (merge topics + risk)
conv_final = pd.merge(
    conv_topics,
    conv_risk[['OpenchannelInteractionId', 'mean_risk', 'max_risk', 'high_risk_flag']],
    on='OpenchannelInteractionId'
)
conv_final.to_excel('conversations_topics_risk.xlsx', index=False)

print('Exported:')
print('  messages_topics_risk.xlsx      — one row per message')
print('  conversations_topics_risk.xlsx — one row per conversation')

## 10. Inspect High-Risk Conversations

In [ ]:
high_risk_ids = conv_risk[conv_risk['high_risk_flag']]['OpenchannelInteractionId'].tolist()

print(f'High-risk conversation IDs: {high_risk_ids[:10]} ...')

# Show top-5 riskiest messages across all conversations
top_risk_msgs = (
    df_in[['OpenchannelInteractionId', 'body_anonimizado', 'dominant_topic', 'bert_risk_score']]
    .sort_values('bert_risk_score', ascending=False)
    .head(10)
)
pd.set_option('display.max_colwidth', 80)
top_risk_msgs